In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [9]:
# Q16: Service combination with lowest churn
combos = df.groupby(['InternetService', 'TechSupport'])['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index()
combos.columns = ['InternetService', 'TechSupport', 'churn_rate']
combos.sort_values('churn_rate')

,InternetService,TechSupport,churn_rate
4,No,No internet service,7.40
1,DSL,Yes,9.68
3,Fiber optic,Yes,22.63
0,DSL,No,27.76
2,Fiber optic,No,49.37


In [ ]:
# Q17: Services most strongly associated with churn reduction
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

results = []
for service in service_cols:
    churn_with = round((df[df[service] == 'Yes']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'Yes']) * 100, 2)
    churn_without = round((df[df[service] == 'No']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'No']) * 100, 2)
    reduction = round(churn_without - churn_with, 2)

    results.append({
        'service': service,
        'churn_with': churn_with,
        'churn_without': churn_without,
        'churn_reduction': reduction
    })

pd.DataFrame(results).sort_values('churn_reduction', ascending=False)

,service,churn_with,churn_without,churn_reduction
0,OnlineSecurity,14.61,41.77,27.16
3,TechSupport,15.17,41.64,26.47
1,OnlineBackup,21.53,39.93,18.40
2,DeviceProtection,22.50,39.13,16.63
5,StreamingMovies,29.94,33.68,3.74
4,StreamingTV,30.07,33.52,3.45


In [4]:
# Q18: Build a Service Adoption Score and analyze its relationship with churn
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['adoption_score'] = df[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

df.groupby('adoption_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})

,adoption_score,churn_rate
0,0,21.41
1,1,45.76
2,2,35.82
3,3,27.37
4,4,22.30
5,5,12.43
6,6,5.28


In [5]:
# Q19: Fiber optic, no tech support, no online security — churn rate
high_risk = df[
    (df['InternetService'] == 'Fiber optic') &
    (df['TechSupport'] == 'No') &
    (df['OnlineSecurity'] == 'No')
]

print(f'Customers matching criteria: {len(high_risk)}')
print(f'Churned: {(high_risk["Churn"] == "Yes").sum()}')
print(f'Churn rate: {round((high_risk["Churn"] == "Yes").sum() / len(high_risk) * 100, 2)}%')

Customers matching criteria: 1765
Churned: 971
Churn rate: 55.01%


In [ ]:
# Q20: Most profitable service bundle
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['bundle'] = df[service_cols].apply(
    lambda row: '+'.join([col for col in service_cols if row[col] == 'Yes']), axis=1
)

df['bundle'] = df['bundle'].replace('', 'No Services')

bundle_analysis = df.groupby('bundle').agg(
    customer_count = ('customerID', 'count'),
    avg_monthly = ('MonthlyCharges', lambda x: round(x.mean(), 2)),
    churn_rate = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)),
    avg_clv = ('TotalCharges', lambda x: round(x.mean(), 2))
).reset_index()

bundle_analysis = bundle_analysis[bundle_analysis['customer_count'] >= 30]

bundle_analysis.sort_values('avg_clv', ascending=False).head(10)

,bundle,customer_count,avg_monthly,churn_rate,avg_clv
42,OnlineSecurity+OnlineBackup+DeviceProtection+T...,284,99.37,5.28,6466.92
38,OnlineSecurity+OnlineBackup+DeviceProtection+S...,92,97.82,14.13,5586.91
17,OnlineBackup+DeviceProtection+TechSupport+Stre...,169,97.40,15.38,5485.96
49,OnlineSecurity+OnlineBackup+TechSupport+Stream...,59,95.14,18.64,5114.52
37,OnlineSecurity+OnlineBackup+DeviceProtection+S...,31,88.08,16.13,4925.67
33,OnlineSecurity+DeviceProtection+TechSupport+St...,131,89.60,10.69,4857.96
40,OnlineSecurity+OnlineBackup+DeviceProtection+T...,63,82.15,6.35,4748.17
24,OnlineBackup+TechSupport+StreamingTV+Streaming...,74,94.80,14.86,4700.60
13,OnlineBackup+DeviceProtection+StreamingTV+Stre...,196,97.94,39.29,4575.37
45,OnlineSecurity+OnlineBackup+StreamingTV+Stream...,44,94.64,22.73,4573.74
